In [ ]:
# 05_explain_lime.ipynb
# Post-hoc Explainability using LIME

import sys
sys.path.append("../src")
from utils import DATASETS, TARGET, RANDOM_STATE, N_INSTANCES


from lime.lime_tabular import LimeTabularExplainer
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

processed_path = Path("../datasets/processed")
models_path = Path("../results/black_box_models")
lime_results_path = Path("../results/LIME")
lime_results_path.mkdir(parents=True, exist_ok=True)

for ds in DATASETS:
    print(f"\nDATASET: {ds}")

    data = pd.read_csv(processed_path / f"{ds}_processed.csv")
    X = data.drop(columns=[TARGET])
    y = data[TARGET]

    s = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
    train_idx, test_idx = next(s.split(X, y))
    X_train = X.iloc[train_idx].reset_index(drop=True)
    X_test  = X.iloc[test_idx].reset_index(drop=True)
    y_test  = y.iloc[test_idx].reset_index(drop=True)

    explainer = LimeTabularExplainer(
        training_data=X_train.values,
        feature_names=X.columns.tolist(),
        class_names=["No Defect", "Defect"],
        mode="classification",
        discretize_continuous=True,
        random_state=RANDOM_STATE
    )

    defect_idx = np.where(y_test.values == 1)[0]
    if len(defect_idx) == 0:
        print(f"  No defective instances in {ds}")
        continue
    selected_idx = defect_idx[:N_INSTANCES]

    dataset_model_path = models_path / ds
    if not dataset_model_path.exists():
        continue

    for model_dir in sorted(dataset_model_path.iterdir()):
        if not model_dir.is_dir():
            continue

        model_name = model_dir.name
        print(f" → {model_name}")

        model = joblib.load(model_dir / "model.joblib")

        save_path = lime_results_path / ds / model_name
        save_path.mkdir(parents=True, exist_ok=True)

        records = []

        for i, idx in enumerate(selected_idx):
            instance = X_test.iloc[idx].values

            explanation = explainer.explain_instance(
                data_row=instance,
                predict_fn=model.predict_proba
            )

            html_file = save_path / f"lime_instance_{i}.html"
            explanation.save_to_file(str(html_file))

